# Explore Chen 2024 NWB files

Use this notebook to inspect subjects/sessions before preprocessing.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO

In [2]:
RAW_DIR = Path("../raw")  # change if needed
nwb_paths = sorted(RAW_DIR.rglob("*.nwb"))
len(nwb_paths)

0

In [3]:
def summarize_file(path: Path):
    with NWBHDF5IO(str(path), "r", load_namespaces=True) as io:
        nwb = io.read()
        units = nwb.units.to_dataframe()
        trials = nwb.trials.to_dataframe()
        subject_id = str(getattr(nwb.subject, "subject_id", "unknown")).removeprefix("sub-")
        session_id = str(nwb.session_id or path.stem)
        trial_durations = (trials["stop_time"] - trials["start_time"]).to_numpy(dtype=float)
        total_time = float(trial_durations.sum()) if len(trial_durations) else np.nan

        annotations = units.get("anno_name", pd.Series([], dtype=object)).astype(str)
        annotations = annotations[annotations.str.len() > 0]
        region_counts = annotations.value_counts()

        unit_class = units.get("classification", pd.Series([], dtype=object)).astype(str)
        good_mask = unit_class.eq("good") if len(unit_class) else pd.Series([], dtype=bool)
        n_good = int(good_mask.sum()) if len(good_mask) else 0
        good_fraction = n_good / len(units) if len(units) else np.nan

        rates = []
        if total_time and not np.isnan(total_time) and total_time > 0 and "spike_times" in units:
            for spikes in units["spike_times"]:
                rates.append(len(spikes) / total_time)
        mean_rate_hz = float(np.mean(rates)) if rates else np.nan

        return {
            "subject_id": subject_id,
            "session_id": session_id,
            "source_file": str(path),
            "n_trials": int(len(trials)),
            "n_units": int(len(units)),
            "n_good_units": n_good,
            "good_unit_fraction": good_fraction,
            "n_regions": int(region_counts.size),
            "mean_firing_rate_hz": mean_rate_hz,
            "region_counts": region_counts.to_dict(),
        }

summary_rows = [summarize_file(p) for p in nwb_paths]
summary = pd.DataFrame(summary_rows)
summary

""


In [ ]:
# Which session has the most regions?
summary.sort_values("n_regions", ascending=False).head(10)[
    ["subject_id", "session_id", "n_regions", "n_units", "n_good_units", "mean_firing_rate_hz"]
]

In [ ]:
def regions_table(summary_df: pd.DataFrame, min_neurons: int = 20):
    rows = []
    for _, row in summary_df.iterrows():
        for region, count in row["region_counts"].items():
            if count >= min_neurons:
                rows.append({
                    "subject_id": row["subject_id"],
                    "session_id": row["session_id"],
                    "region": region,
                    "n_neurons": int(count),
                })
    return pd.DataFrame(rows).sort_values(["n_neurons", "subject_id", "session_id"], ascending=[False, True, True])

regions_table(summary, min_neurons=30).head(50)

After exploring, copy selected subject IDs and trial metadata fields into a config YAML and run `scripts/chen2024/preprocess.py`.